# 08 — XGBoost Model

This notebook implements conservative regularized XGBoost regression with the frozen target,
features, horizon, and chronological partitions. If XGBoost is unavailable, execution stops
with a clear dependency message rather than substituting another algorithm.


## 1. Inputs and dependencies


In [ ]:
from pathlib import Path
import importlib.util
import json
import pickle
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook", rc={"figure.dpi": 120, "savefig.dpi": 300})

def locate_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "phase4_utils.py").exists():
            return candidate
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = locate_root(Path.cwd())
spec = importlib.util.spec_from_file_location("phase4_utils", PROJECT_ROOT / "src" / "phase4_utils.py")
u = importlib.util.module_from_spec(spec)
spec.loader.exec_module(u)
evaluation = u.load_evaluation_module(PROJECT_ROOT)
splits = u.load_splits(PROJECT_ROOT)
print({name: data["combined"].shape for name, data in splits.items()})


In [ ]:
for split_name, data in splits.items():
    assert data["X"][u.KEYS].equals(data["y"][u.KEYS])
    assert sorted(data["X"]["year"].unique().tolist()) == u.SPLIT_YEARS[split_name]
    assert data["X"]["region"].nunique() == 13
    assert not data["X"].duplicated(u.KEYS).any()
    assert list(data["X"].columns) == u.KEYS + u.PREDICTORS
print("Frozen protocol and target alignment verified.")


In [ ]:
try:
    import xgboost
    import sklearn
    import joblib
    from xgboost import XGBRegressor
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.pipeline import Pipeline
except ImportError as exc:
    raise ImportError("Notebook 08 requires xgboost, scikit-learn, and joblib.") from exc

MODEL_DIR=PROJECT_ROOT/"models"/"xgboost"; RESULT_DIR=PROJECT_ROOT/"results"/"xgboost"
FIGURE_DIR=PROJECT_ROOT/"figures"/"xgboost"
for directory in (MODEL_DIR,RESULT_DIR,FIGURE_DIR): directory.mkdir(parents=True,exist_ok=True)
u.write_json(RESULT_DIR/"environment.json",u.package_versions(
    ["numpy","pandas","sklearn","xgboost","joblib","matplotlib","seaborn"]))

MODEL_COLUMNS=["region"]+u.PREDICTORS
def make_pipeline(params):
    preprocess=ColumnTransformer([
        ("region",OneHotEncoder(handle_unknown="ignore",sparse_output=False),["region"]),
        ("numeric","passthrough",u.PREDICTORS)],remainder="drop")
    model=XGBRegressor(objective="reg:squarederror",random_state=u.RANDOM_SEED,
        n_jobs=1,verbosity=0,**params)
    return Pipeline([("preprocess",preprocess),("model",model)])

CANDIDATES=[
 {"n_estimators":50,"max_depth":1,"learning_rate":0.05,"min_child_weight":3,"subsample":0.8,"colsample_bytree":0.8,"reg_alpha":0.1,"reg_lambda":10.0},
 {"n_estimators":100,"max_depth":1,"learning_rate":0.05,"min_child_weight":3,"subsample":0.8,"colsample_bytree":0.8,"reg_alpha":0.1,"reg_lambda":10.0},
 {"n_estimators":75,"max_depth":2,"learning_rate":0.03,"min_child_weight":4,"subsample":0.8,"colsample_bytree":0.8,"reg_alpha":0.5,"reg_lambda":15.0},
 {"n_estimators":100,"max_depth":2,"learning_rate":0.03,"min_child_weight":5,"subsample":0.75,"colsample_bytree":0.75,"reg_alpha":1.0,"reg_lambda":20.0},
]


## 2. Validation selection and fixed test evaluation


In [ ]:
train,val,test=splits["train"]["combined"],splits["validation"]["combined"],splits["test"]["combined"]
rows=[]
for index,params in enumerate(CANDIDATES):
    pipeline=make_pipeline(params); pipeline.fit(train[MODEL_COLUMNS],train[u.TARGET])
    predicted=pipeline.predict(val[MODEL_COLUMNS])
    metric=evaluation.evaluate_regression(val[u.TARGET],predicted,
        model_name=f"XGB_candidate_{index+1}",split="validation")
    metric["candidate_id"]=index+1; rows.append(metric)
candidate_metrics=pd.DataFrame(rows).sort_values(["RMSE","candidate_id"]).reset_index(drop=True)
selected_id=int(candidate_metrics.iloc[0]["candidate_id"]); selected_params=CANDIDATES[selected_id-1]

validation_model=make_pipeline(selected_params); validation_model.fit(train[MODEL_COLUMNS],train[u.TARGET])
vp=validation_model.predict(val[MODEL_COLUMNS])
validation_output=u.prediction_frame(val[u.KEYS],val[u.TARGET],vp,"XGBoost","validation")

development=pd.concat([train,val],ignore_index=True).sort_values(u.KEYS)
final_model=make_pipeline(selected_params); final_model.fit(development[MODEL_COLUMNS],development[u.TARGET])
tp=final_model.predict(test[MODEL_COLUMNS])
test_output=u.prediction_frame(test[u.KEYS],test[u.TARGET],tp,"XGBoost","test")
predictions=pd.concat([validation_output,test_output],ignore_index=True)
metrics=pd.DataFrame([u.evaluate_prediction_frame(evaluation,validation_output),
                      u.evaluate_prediction_frame(evaluation,test_output)])
predictions.to_csv(RESULT_DIR/"predictions.csv",index=False,float_format="%.15g")
metrics.to_csv(RESULT_DIR/"metrics.csv",index=False,float_format="%.15g")
candidate_metrics.to_csv(RESULT_DIR/"candidate_validation_metrics.csv",index=False)
joblib.dump(validation_model,MODEL_DIR/"validation_model.joblib")
joblib.dump(final_model,MODEL_DIR/"xgboost_model.joblib")
u.write_json(MODEL_DIR/"parameters.json",{"random_seed":u.RANDOM_SEED,
 "selected_candidate_id":selected_id,"parameters":selected_params,
 "selection_split":"validation","test_refit_years":[2019,2020,2021]})
names=final_model.named_steps["preprocess"].get_feature_names_out()
importance=pd.DataFrame({"feature":names,
 "importance":final_model.named_steps["model"].feature_importances_}).sort_values("importance",ascending=False)
importance.to_csv(RESULT_DIR/"feature_importance.csv",index=False)
display(candidate_metrics); display(metrics); display(importance.head(15))


## 3. Figures and summary


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
for ax,split_name in zip(axes,["validation","test"]):
 group=predictions.query("dataset_split == @split_name").sort_values("actual")
 ax.plot(group["actual"].to_numpy()/1e9,marker="o",label="Observed")
 ax.plot(group["predicted"].to_numpy()/1e9,marker="s",label="XGBoost")
 ax.set_title(split_name.title()); ax.set_ylabel("Billion kWh")
axes[1].legend(frameon=False); fig.tight_layout()
fig.savefig(FIGURE_DIR/"predictions.png",bbox_inches="tight"); plt.show()

fig,ax=plt.subplots(figsize=(9,5)); sns.scatterplot(data=predictions,x="predicted",y="residual",
 hue="dataset_split",ax=ax); ax.axhline(0,color="black",linewidth=1)
ax.set_title("XGBoost Residuals",weight="bold"); fig.tight_layout()
fig.savefig(FIGURE_DIR/"residuals.png",bbox_inches="tight"); plt.show()

top=importance.head(15).sort_values("importance")
fig,ax=plt.subplots(figsize=(10,7)); sns.barplot(data=top,x="importance",y="feature",color="#6A4C93",ax=ax)
ax.set_title("XGBoost Feature Importance",weight="bold"); fig.tight_layout()
fig.savefig(FIGURE_DIR/"feature_importance.png",bbox_inches="tight"); plt.show()

report=f'''# XGBoost Summary

- Random seed: {u.RANDOM_SEED}
- Candidate configurations: {len(CANDIDATES)}
- Selected using: 2021 validation RMSE
- Final refit: 2019–2021
- Test: 2022 once

Shallow trees, shrinkage, row/column subsampling, minimum child weight, and L1/L2
regularization constrain flexibility. The sample remains very small; importance values are
predictive diagnostics and do not imply causation.
'''
(RESULT_DIR/"summary_report.md").write_text(report,encoding="utf-8")
print("Notebook 08 complete.")
